# 📥 LogShield — Notebook 01: Multi-Format Log Ingestion & Validation

This notebook demonstrates how LogShield ingests multi-format enterprise web server access logs (Apache CLF, NGINX, and structured JSON), validates IP addresses and HTTP status codes, and isolates malformed records to the quarantine directory.

In [ ]:
from pathlib import Path
from src.logshield.ingestion.apache_parser import parse_apache_line
from src.logshield.ingestion.nginx_parser import parse_nginx_line
from src.logshield.ingestion.json_parser import parse_json_line
from src.logshield.ingestion.validator import validate_record, is_valid_ip, is_valid_status_code

print("[✓] LogShield Ingestion modules imported successfully.")

## 1. Parse Sample Apache Common Log Format (CLF) Record

In [ ]:
sample_line = '10.0.12.45 - - [01/Sep/2026:03:14:22 +0000] "GET /admin/login HTTP/1.1" 404 342'
record = parse_apache_line(sample_line)
print("IP:", record.ip)
print("Method:", record.method)
print("Path:", record.path)
print("Status:", record.status)
print("Bytes:", record.response_bytes)
print("Timestamp:", record.timestamp)
print("Partition Date:", record.date, "Hour:", record.hour)

## 2. Validate Quality and Check Data Hygiene

In [ ]:
is_valid, reason = validate_record(record)
print(f"Record Valid: {is_valid} (Error Reason: {reason})")

## 3. Streaming Log File Ingestion with Quarantine Isolation

In [ ]:
from src.logshield.ingestion.log_reader import LogReader

reader = LogReader()
log_path = Path("server.log")

records = []
for i, rec in enumerate(reader.read_file(log_path)):
    records.append(rec)
    if i >= 1000:
        break

print(f"Streamed {len(records)} sample records.")
print("Reader Ingestion Stats:", reader.stats)